# Model Stability Audit: Repeated Customer-Grouped Splits

This notebook measures how stable the results of the six experiment cells are (three feature sets, two releases) when the train and test partitions change.

Each cell was evaluated on one customer-grouped split (seed 42) whose test partition has only 59 rows, so one test row moves a metric by about 1.7 percentage points. A single split cannot show how much of a difference between two cells is ordinary split-to-split variation. Here the partitioning is repeated with 50 different seeds, the model is refitted on every split, and the variation of each metric is reported.

## Design

- **Splits.** For each of the 50 seeds (1 to 51, without 42, the seed of the modelling notebooks) the table is partitioned as in the modelling notebooks: two successive grouped shuffle splits by `Customer_ID` (70% train, then the remainder halved into validation and test), using the source customer labels. The same partitions are applied to both releases through `Property_ID`.
- **Model.** One model, with no tuning and no selection: the Random Forest of notebook 03 (500 trees, balanced subsample class weights, seed 42, all other settings at their defaults). Its predicted class gives Top-1 accuracy, Macro F1 and Weighted F1, and its ranked class probabilities give Top-2 and Top-3 accuracy. It is refitted on the training partition of every split.
- **Final configuration of the ranking forest.** The ranking forest saved by the modelling notebooks, whose setting was selected by a search, is also refitted on every split without re-tuning. For each feature set the setting of the source cell is used for both releases, so that the paired comparison between releases does not mix different settings. Its Top-2 and Top-3 are reported next to those of the default forest.
- **Reference.** A class-prior baseline ranks the classes by their frequency in the training partition.
- **Paired comparisons.** Differences are computed within each split: anonymised release minus source release for each feature set, the full feature set minus the no-AI feature set (with AI minus without AI) and the pure-record feature set minus the full feature set within each release.
- **What is reported.** Mean, standard deviation and the 2.5th and 97.5th percentiles over the 50 splits, and the values are descriptive. The splits are drawn from the same 431 states and overlap, so the spread is not the sampling variability of a new set of customers, and no significance statement is made. The test partitions differ in size from split to split (about 52 to 83 rows). The settings of the saved ranking forests were selected by searches on the seed-42 split, so Top-2 and Top-3 obtained with them can be somewhat optimistic, while the default forest involves no selection.
- **Setting sensitivity.** The setting of the ranking forest is not chosen. Its influence on Top-2 and Top-3 is shown in a separate section by refitting it with every combination of the values selected by the six searches of the modelling notebooks (12 settings).
- **Where the results are kept.** Every table is printed in the output of the cell that computes it, the interpretation at the end of this notebook summarises them, and no result files are written.
- **Difference from the modelling notebooks.** The modelling notebooks use a regularised Random Forest, selected by a search, for Top-2 and Top-3. Here the default forest gives all main metrics, so that no setting has to be chosen, and the final configuration of the ranking forest is added as a second source of Top-2 and Top-3. The Top-2 and Top-3 of the default forest are therefore not comparable with the saved single-split values, but Top-1 accuracy, Macro F1 and Weighted F1 are.

In [1]:
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score, f1_score
from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

warnings.filterwarnings("ignore")

DATA_DIR = Path("../../data/processed/hypothesis_testing")

SEEDS = [seed for seed in range(1, 52) if seed != 42]
CHECK_SEED = 42

## Data

Both modelling tables have the same rows in the same order. The check below confirms that the keys, timestamps and targets are identical in the two releases, so the same row indices can be used for both.

In [2]:
source = pd.read_csv(DATA_DIR / "source_dataset" / "model_state.csv", parse_dates=["Prediction_Time"])
regions = pd.read_csv(DATA_DIR / "regions_anonymised" / "model_state.csv", parse_dates=["Prediction_Time"])

for column in ["Property_ID", "Prediction_Time", "Next_Category"]:
    assert (source[column].values == regions[column].values).all(), column

RELEASES = {"source": source, "regions": regions}

print("DATA LOADED")
print("-" * 60)
print(f"Rows:       {len(source):,}")
print(f"Properties: {source['Property_ID'].nunique():,}")
print(f"Customers:  {source['Customer_ID'].nunique():,}")

DATA LOADED
------------------------------------------------------------
Rows:       431
Properties: 234
Customers:  198


## Feature Sets And Model

The three feature sets are the ones used in the modelling notebooks: all 55 predictors, the 42 predictors without the `ai_` columns, and the 12 documented-record predictors. The model is the Random Forest of notebook 03.

In [3]:
IDENTIFIERS = ["Property_ID", "Customer_ID", "Prediction_Time", "Next_Category_ID", "Next_Category"]

ALL_FEATURES = [column for column in source.columns if column not in IDENTIFIERS]
NO_AI_FEATURES = [column for column in ALL_FEATURES if not column.startswith("ai_")]
RECORD_FEATURES = [
    "documented_asset_count",
    "documented_category_count",
    "documented_value_total",
    "documented_replacement_cost_total",
    "mean_evidence_rating",
    "documented_count_CAT-001",
    "documented_count_CAT-002",
    "documented_count_CAT-003",
    "documented_count_CAT-004",
    "documented_count_CAT-005",
    "documented_count_CAT-006",
    "documented_count_CAT-007",
]

FEATURE_SETS = {
    "full": ALL_FEATURES,
    "no_ai": NO_AI_FEATURES,
    "pure_record": RECORD_FEATURES,
}

assert [len(columns) for columns in FEATURE_SETS.values()] == [55, 42, 12]

# The Random Forest of notebook 03: no search, no tuning.
FOREST_SETTINGS = dict(
    n_estimators=500,
    class_weight="balanced_subsample",
    random_state=42,
    n_jobs=-1,
)

print("FEATURE SETS")
print("-" * 60)
for name, columns in FEATURE_SETS.items():
    print(f"{name:<12} {len(columns)} predictors")

FEATURE SETS
------------------------------------------------------------
full         55 predictors
no_ai        42 predictors
pure_record  12 predictors


## Check Against The Saved Models

The feature sets and the forest above are written out in this notebook. The cell below loads the six saved Top-1 models of the modelling notebooks and confirms that each one was trained on exactly these predictors and with exactly these forest settings, so that nothing in this notebook rests on an unchecked assumption.

In [4]:
import joblib

MODELS_DIR = Path("../../models/hypothesis_testing")

print("CHECK AGAINST THE SAVED TOP-1 MODELS")
print("-" * 80)

for release in RELEASES:
    for feature_set, columns in FEATURE_SETS.items():
        saved = joblib.load(MODELS_DIR / f"{release}_{feature_set}_top1.joblib")

        saved_columns = [
            column
            for name, _, group in saved.named_steps["preprocessor"].transformers_
            if name in ("numerical", "categorical")
            for column in group
        ]

        saved_forest = saved.named_steps["classifier"].get_params()

        assert set(saved_columns) == set(columns), (release, feature_set)
        assert len(saved_columns) == len(columns), (release, feature_set)

        for name, value in FOREST_SETTINGS.items():
            assert saved_forest[name] == value, (release, feature_set, name)

        print(
            f"{release:<8} {feature_set:<12} predictors {len(saved_columns):>2} match | "
            f"forest settings match ({saved_forest['n_estimators']} trees)"
        )

print()
print("The saved Top-1 models use exactly the predictors and settings defined above.")

CHECK AGAINST THE SAVED TOP-1 MODELS
--------------------------------------------------------------------------------
source   full         predictors 55 match | forest settings match (500 trees)
source   no_ai        predictors 42 match | forest settings match (500 trees)
source   pure_record  predictors 12 match | forest settings match (500 trees)
regions  full         predictors 55 match | forest settings match (500 trees)
regions  no_ai        predictors 42 match | forest settings match (500 trees)
regions  pure_record  predictors 12 match | forest settings match (500 trees)

The saved Top-1 models use exactly the predictors and settings defined above.


## Functions

The preprocessing is the one of the modelling notebooks (median imputation and scaling for numeric columns, mode imputation and one-hot encoding for the others) and is fitted on the training partition only. Top-k accuracy is computed from the ranked class probabilities, and a true class that is absent from a training partition counts as a miss.

In [5]:
def build_pipeline(frame, columns):
    numeric = frame[columns].select_dtypes(include=["number"]).columns.tolist()
    categorical = [column for column in columns if column not in numeric]

    preprocessor = ColumnTransformer(
        transformers=[
            (
                "numerical",
                Pipeline([("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler())]),
                numeric,
            ),
            (
                "categorical",
                Pipeline([
                    ("imputer", SimpleImputer(strategy="most_frequent")),
                    ("encoder", OneHotEncoder(handle_unknown="ignore")),
                ]),
                categorical,
            ),
        ]
    )

    return Pipeline([
        ("preprocessor", preprocessor),
        ("classifier", RandomForestClassifier(**FOREST_SETTINGS)),
    ])


def top_k_accuracy(model, X, y, k):
    probabilities = model.predict_proba(X)
    classes = model.named_steps["classifier"].classes_
    ranked = classes[np.argsort(-probabilities, axis=1, kind="stable")[:, :k]]
    return float(np.mean([truth in row for truth, row in zip(y, ranked)]))


def customer_grouped_split(seed):
    groups = source["Customer_ID"]
    target = source["Next_Category"]

    train_idx, temp_idx = next(
        GroupShuffleSplit(n_splits=1, train_size=0.70, random_state=seed).split(
            source, target, groups=groups
        )
    )

    val_rel, test_rel = next(
        GroupShuffleSplit(n_splits=1, train_size=0.50, random_state=seed).split(
            source.iloc[temp_idx], target.iloc[temp_idx], groups=groups.iloc[temp_idx]
        )
    )

    return train_idx, temp_idx[val_rel], temp_idx[test_rel]


def class_prior_baseline(y_train, y_test):
    counts = y_train.value_counts()
    order = sorted(counts.index, key=lambda label: (-counts[label], label))

    predictions = np.repeat(order[0], len(y_test))

    return dict(
        top1_accuracy=accuracy_score(y_test, predictions),
        macro_f1=f1_score(y_test, predictions, average="macro", zero_division=0),
        weighted_f1=f1_score(y_test, predictions, average="weighted", zero_division=0),
        top2_accuracy=float(np.mean([truth in order[:2] for truth in y_test])),
        top3_accuracy=float(np.mean([truth in order[:3] for truth in y_test])),
    )


def ranking_pipeline(frame, columns, settings):
    pipeline = build_pipeline(frame, columns)
    pipeline.set_params(**{f"classifier__{name}": value for name, value in settings.items()})
    return pipeline


def evaluate(release, feature_set, train_idx, test_idx, with_final=False):
    frame = RELEASES[release]
    columns = FEATURE_SETS[feature_set]
    X_train, X_test = frame[columns].iloc[train_idx], frame[columns].iloc[test_idx]
    y_train, y_test = frame["Next_Category"].iloc[train_idx], frame["Next_Category"].iloc[test_idx]

    model = build_pipeline(frame, columns).fit(X_train, y_train)
    predictions = model.predict(X_test)

    result = dict(
        top1_accuracy=accuracy_score(y_test, predictions),
        macro_f1=f1_score(y_test, predictions, average="macro", zero_division=0),
        weighted_f1=f1_score(y_test, predictions, average="weighted", zero_division=0),
        top2_accuracy=top_k_accuracy(model, X_test, y_test.values, 2),
        top3_accuracy=top_k_accuracy(model, X_test, y_test.values, 3),
    )

    if with_final:
        ranking = ranking_pipeline(frame, columns, FINAL_SETTINGS[feature_set]).fit(X_train, y_train)
        result["final_top2_accuracy"] = top_k_accuracy(ranking, X_test, y_test.values, 2)
        result["final_top3_accuracy"] = top_k_accuracy(ranking, X_test, y_test.values, 3)

    return result

## Reproduction Check

Before the repetitions, the procedure is run on the seed-42 split used in the modelling notebooks. The Top-1 metrics of every cell must equal the holdout results saved by the corresponding notebook, which confirms that the splits, preprocessing and Top-1 model are the same.

In [6]:
FEATURE_FOLDERS = {
    "full": "01_full_feature_set",
    "no_ai": "02_no_ai_features",
    "pure_record": "03_pure_record",
}
RELEASE_FOLDERS = {"source": "source_dataset", "regions": "regions_anonymised"}

train_idx, val_idx, test_idx = customer_grouped_split(CHECK_SEED)

print("REPRODUCTION OF THE SEED-42 TOP-1 RESULTS")
print("-" * 80)

for release, release_folder in RELEASE_FOLDERS.items():
    for feature_set, feature_folder in FEATURE_FOLDERS.items():
        saved = pd.read_csv(
            DATA_DIR / release_folder / feature_folder / "results_holdout_test.csv"
        ).iloc[0]

        result = evaluate(release, feature_set, train_idx, test_idx)

        differences = {
            "top1_accuracy": abs(result["top1_accuracy"] - saved["Top1_Accuracy"]),
            "macro_f1": abs(result["macro_f1"] - saved["Top1_Macro_F1"]),
            "weighted_f1": abs(result["weighted_f1"] - saved["Top1_Weighted_F1"]),
        }

        print(
            f"{release:<8} {feature_set:<12} "
            f"accuracy {result['top1_accuracy']:.4f} | Macro F1 {result['macro_f1']:.4f} | "
            f"max difference to saved {max(differences.values()):.2e}"
        )

        assert max(differences.values()) < 1e-9, (release, feature_set)

print()
print("All Top-1 metrics reproduce the saved holdout results.")

REPRODUCTION OF THE SEED-42 TOP-1 RESULTS
--------------------------------------------------------------------------------
source   full         accuracy 0.4915 | Macro F1 0.3683 | max difference to saved 5.55e-17
source   no_ai        accuracy 0.5424 | Macro F1 0.4085 | max difference to saved 5.55e-17
source   pure_record  accuracy 0.4576 | Macro F1 0.3452 | max difference to saved 0.00e+00
regions  full         accuracy 0.4915 | Macro F1 0.3620 | max difference to saved 5.55e-17
regions  no_ai        accuracy 0.4915 | Macro F1 0.3380 | max difference to saved 5.55e-17
regions  pure_record  accuracy 0.4576 | Macro F1 0.3452 | max difference to saved 0.00e+00

All Top-1 metrics reproduce the saved holdout results.


## Final Configuration Of The Ranking Forest

The modelling notebooks selected the setting of the ranking forest by a random search in each of the six cells and saved the selected forest. The cell below loads the six saved ranking forests and reads the setting that each one actually uses, so that no setting is typed by hand. The searches chose different settings in different cells and reached similar cross-validation Macro F1 scores (0.39 to 0.44, as reported in the modelling notebooks), so no setting stood out.

For the repeated splits, the final configuration of each feature set is the setting of its **source** cell, used for both releases.

In [7]:
import joblib

MODELS_DIR = Path("../../models/hypothesis_testing")

PARAMETERS = ["n_estimators", "min_samples_split", "min_samples_leaf", "max_features", "max_depth"]

selected_rows = []

for release in RELEASE_FOLDERS:
    for feature_set in FEATURE_FOLDERS:
        saved_forest = joblib.load(MODELS_DIR / f"{release}_{feature_set}_ranking.joblib")
        forest = saved_forest.named_steps["classifier"].get_params()
        selected_rows.append(dict(release=release, feature_set=feature_set, **{name: forest[name] for name in PARAMETERS}))

print("SETTINGS OF THE SAVED RANKING FORESTS (read from the saved models)")
print("-" * 100)
print(pd.DataFrame(selected_rows).to_string(index=False))

FINAL_SETTINGS = {
    row["feature_set"]: {name: row[name] for name in PARAMETERS}
    for row in selected_rows
    if row["release"] == "source"
}

print()
print("FINAL CONFIGURATION USED FOR BOTH RELEASES (setting of the source cell)")
print("-" * 100)
for feature_set, settings in FINAL_SETTINGS.items():
    print(f"{feature_set:<12} {settings}")

SETTINGS OF THE SAVED RANKING FORESTS (read from the saved models)
----------------------------------------------------------------------------------------------------
release feature_set  n_estimators  min_samples_split  min_samples_leaf  max_features  max_depth
 source        full           300                 15                 1           0.5        7.0
 source       no_ai           300                  2                 1           0.5        NaN
 source pure_record           300                  5                 6           0.5        7.0
regions        full           300                  2                 1           0.5        NaN
regions       no_ai           300                  2                 1           0.5        NaN
regions pure_record           300                  5                 6           0.5        7.0

FINAL CONFIGURATION USED FOR BOTH RELEASES (setting of the source cell)
---------------------------------------------------------------------------------------

## Repeated Splits

For each of the 50 seeds the model is refitted on the training partition and scored on the test partition, for both releases and all three feature sets. Nothing is tuned or selected on any split.

In [8]:
records = []
baselines = []

for seed in SEEDS:
    train_idx, val_idx, test_idx = customer_grouped_split(seed)

    y_train = source["Next_Category"].iloc[train_idx]
    y_test = source["Next_Category"].iloc[test_idx]

    baselines.append(dict(seed=seed, **class_prior_baseline(y_train, y_test)))

    for release in RELEASES:
        for feature_set in FEATURE_SETS:
            records.append(
                dict(
                    seed=seed,
                    release=release,
                    feature_set=feature_set,
                    test_rows=len(test_idx),
                    test_customers=int(source["Customer_ID"].iloc[test_idx].nunique()),
                    **evaluate(release, feature_set, train_idx, test_idx, with_final=True),
                )
            )

per_split = pd.DataFrame(records)
baseline_per_split = pd.DataFrame(baselines)


print("REPEATED SPLITS COMPLETE")
print("-" * 60)
print(f"Splits:               {len(SEEDS)}")
print(f"Result rows:          {len(per_split)}")
print(
    f"Test rows per split:  min {per_split['test_rows'].min()} | "
    f"mean {per_split['test_rows'].mean():.1f} | max {per_split['test_rows'].max()}"
)

REPEATED SPLITS COMPLETE
------------------------------------------------------------
Splits:               50
Result rows:          300
Test rows per split:  min 52 | mean 66.9 | max 83


## Spread Of Each Cell

Mean, standard deviation and the 2.5th and 97.5th percentiles of every metric over the 50 splits. The class-prior baseline is the same for both releases because it uses only the training labels.

In [9]:
METRICS = {
    "top1_accuracy": "Top-1 accuracy",
    "macro_f1": "Macro F1",
    "weighted_f1": "Weighted F1",
    "top2_accuracy": "Top-2 accuracy",
    "top3_accuracy": "Top-3 accuracy",
}

FINAL_METRICS = {
    "final_top2_accuracy": "Top-2 accuracy (final configuration)",
    "final_top3_accuracy": "Top-3 accuracy (final configuration)",
}

ALL_METRICS = {**METRICS, **FINAL_METRICS}


def describe(values):
    return pd.Series(
        {
            "mean": values.mean(),
            "sd": values.std(ddof=1),
            "p2.5": values.quantile(0.025),
            "p97.5": values.quantile(0.975),
        }
    )


spread_rows = []

for (release, feature_set), group in per_split.groupby(["release", "feature_set"]):
    for column, label in ALL_METRICS.items():
        spread_rows.append(
            dict(release=release, feature_set=feature_set, metric=label, **describe(group[column]))
        )

for column, label in METRICS.items():
    spread_rows.append(
        dict(release="both", feature_set="class_prior_baseline", metric=label, **describe(baseline_per_split[column]))
    )

spread = pd.DataFrame(spread_rows)

print("SPREAD OVER 50 SPLITS")
print("-" * 100)

for (release, feature_set), group in spread.groupby(["release", "feature_set"], sort=False):
    print(f"\n{release} / {feature_set}")
    print(group.drop(columns=["release", "feature_set"]).round(4).to_string(index=False))

SPREAD OVER 50 SPLITS
----------------------------------------------------------------------------------------------------

regions / full
                              metric   mean     sd   p2.5  p97.5
                      Top-1 accuracy 0.4596 0.0533 0.3550 0.5538
                            Macro F1 0.3549 0.0674 0.2474 0.4956
                         Weighted F1 0.4176 0.0548 0.3197 0.5170
                      Top-2 accuracy 0.7107 0.0383 0.6481 0.7703
                      Top-3 accuracy 0.8415 0.0395 0.7754 0.9204
Top-2 accuracy (final configuration) 0.7369 0.0428 0.6629 0.8192
Top-3 accuracy (final configuration) 0.8521 0.0402 0.7767 0.9229

regions / no_ai
                              metric   mean     sd   p2.5  p97.5
                      Top-1 accuracy 0.4845 0.0458 0.3969 0.5514
                            Macro F1 0.3896 0.0607 0.2964 0.5112
                         Weighted F1 0.4460 0.0464 0.3553 0.5330
                      Top-2 accuracy 0.7321 0.0452 0.6554 0.8149

## Paired Differences Between The Releases

Within each split, the anonymised release minus the source release, for each feature set. A positive value means that the anonymised release scored higher. The last columns give the share of splits with a negative, zero and positive difference.

In [10]:
def paired_differences(a, b, comparison):
    rows = []
    for column, label in ALL_METRICS.items():
        difference = (a.set_index("seed")[column] - b.set_index("seed")[column]).dropna()
        rows.append(
            dict(
                comparison=comparison,
                metric=label,
                mean_difference=difference.mean(),
                sd=difference.std(ddof=1),
                **{"p2.5": difference.quantile(0.025), "p97.5": difference.quantile(0.975)},
                share_negative=float((difference < -1e-12).mean()),
                share_zero=float((difference.abs() <= 1e-12).mean()),
                share_positive=float((difference > 1e-12).mean()),
            )
        )
    return rows


def select(release, feature_set):
    return per_split[(per_split.release == release) & (per_split.feature_set == feature_set)]


release_differences = []
for feature_set in FEATURE_SETS:
    release_differences += paired_differences(
        select("regions", feature_set),
        select("source", feature_set),
        f"regions minus source | {feature_set}",
    )

release_differences = pd.DataFrame(release_differences)

print("PAIRED DIFFERENCES: REGIONS MINUS SOURCE")
print("-" * 110)

for comparison, group in release_differences.groupby("comparison", sort=False):
    print(f"\n{comparison}")
    print(group.drop(columns="comparison").round(4).to_string(index=False))

PAIRED DIFFERENCES: REGIONS MINUS SOURCE
--------------------------------------------------------------------------------------------------------------

regions minus source | full
                              metric  mean_difference     sd    p2.5  p97.5  share_negative  share_zero  share_positive
                      Top-1 accuracy           0.0006 0.0307 -0.0563 0.0543            0.40        0.12            0.48
                            Macro F1           0.0044 0.0339 -0.0535 0.0794            0.44        0.00            0.56
                         Weighted F1           0.0028 0.0338 -0.0524 0.0651            0.42        0.00            0.58
                      Top-2 accuracy          -0.0043 0.0362 -0.0689 0.0649            0.42        0.24            0.34
                      Top-3 accuracy          -0.0053 0.0289 -0.0555 0.0429            0.50        0.16            0.34
Top-2 accuracy (final configuration)           0.0027 0.0292 -0.0560 0.0402            0.40        

## Paired Differences Between Feature Sets

Within each release and each split: the full feature set minus the no-AI feature set (with AI minus without AI), and the pure-record feature set minus the full feature set.

In [11]:
feature_set_differences = []

for release in RELEASES:
    full = select(release, "full")
    feature_set_differences += paired_differences(full, select(release, "no_ai"), f"{release} | with AI minus without AI (full minus no-AI)")
    feature_set_differences += paired_differences(select(release, "pure_record"), full, f"{release} | pure record minus full")

feature_set_differences = pd.DataFrame(feature_set_differences)

print("PAIRED DIFFERENCES: FEATURE SETS WITHIN A RELEASE")
print("-" * 110)

for comparison, group in feature_set_differences.groupby("comparison", sort=False):
    print(f"\n{comparison}")
    print(group.drop(columns="comparison").round(4).to_string(index=False))

PAIRED DIFFERENCES: FEATURE SETS WITHIN A RELEASE
--------------------------------------------------------------------------------------------------------------

source | with AI minus without AI (full minus no-AI)
                              metric  mean_difference     sd    p2.5  p97.5  share_negative  share_zero  share_positive
                      Top-1 accuracy          -0.0162 0.0351 -0.0794 0.0469            0.58        0.20            0.22
                            Macro F1          -0.0274 0.0518 -0.1613 0.0394            0.76        0.00            0.24
                         Weighted F1          -0.0221 0.0360 -0.0884 0.0413            0.76        0.00            0.24
                      Top-2 accuracy          -0.0151 0.0386 -0.0987 0.0586            0.60        0.14            0.26
                      Top-3 accuracy          -0.0046 0.0299 -0.0570 0.0533            0.50        0.20            0.30
Top-2 accuracy (final configuration)          -0.0166 0.0368 -0.0

## Sensitivity To The Ranking-Forest Setting

The modelling notebooks selected the setting of the ranking forest by a random search in each of the six cells and saved the selected forest. The main analysis uses the default forest of notebook 03 for the main metrics, and the final configuration of the ranking forest is reported next to it. To show how much Top-2 and Top-3 depend on the setting, the ranking forest is refitted on the same 50 splits with **every combination of the values that appear in the saved ranking forests** (table above). For each cell the spread of the mean Top-2 and Top-3 across these settings is compared with the split-to-split standard deviation of the default forest, and no setting is selected.

In [12]:
from itertools import product

OBSERVED_VALUES = {
    name: sorted({row[name] for row in selected_rows}, key=lambda value: (value is None, value or 0))
    for name in PARAMETERS
}

RANKING_SETTINGS = [
    dict(zip(PARAMETERS, values))
    for values in product(*OBSERVED_VALUES.values())
]

print("VALUES THAT APPEAR IN THE SAVED FORESTS")
print("-" * 60)
for name, values in OBSERVED_VALUES.items():
    print(f"{name:<20} {values}")

print()
print(f"Combinations: {len(RANKING_SETTINGS)}")

assert len(RANKING_SETTINGS) == 12


def setting_label(settings):
    depth = "none" if settings["max_depth"] is None else settings["max_depth"]
    return f"split {settings['min_samples_split']} / leaf {settings['min_samples_leaf']} / depth {depth}"


sensitivity_rows = []

for seed in SEEDS:
    train_idx, val_idx, test_idx = customer_grouped_split(seed)

    for release, frame in RELEASES.items():
        for feature_set, columns in FEATURE_SETS.items():
            X_train, X_test = frame[columns].iloc[train_idx], frame[columns].iloc[test_idx]
            y_train, y_test = frame["Next_Category"].iloc[train_idx], frame["Next_Category"].iloc[test_idx]

            for settings in RANKING_SETTINGS:
                model = ranking_pipeline(frame, columns, settings).fit(X_train, y_train)

                sensitivity_rows.append(
                    dict(
                        seed=seed,
                        release=release,
                        feature_set=feature_set,
                        setting=setting_label(settings),
                        top2_accuracy=top_k_accuracy(model, X_test, y_test.values, 2),
                        top3_accuracy=top_k_accuracy(model, X_test, y_test.values, 3),
                    )
                )

sensitivity = pd.DataFrame(sensitivity_rows)

print()
print("RANKING-FOREST SETTINGS EVALUATED")
print("-" * 60)
print(f"Settings:     {len(RANKING_SETTINGS)}")
print(f"Splits:       {len(SEEDS)}")
print(f"Result rows:  {len(sensitivity)}")

VALUES THAT APPEAR IN THE SAVED FORESTS
------------------------------------------------------------
n_estimators         [300]
min_samples_split    [2, 5, 15]
min_samples_leaf     [1, 6]
max_features         [0.5]
max_depth            [7, None]

Combinations: 12

RANKING-FOREST SETTINGS EVALUATED
------------------------------------------------------------
Settings:     12
Splits:       50
Result rows:  3600


In [13]:
settings_mean = (
    sensitivity
    .groupby(["release", "feature_set", "setting"])[["top2_accuracy", "top3_accuracy"]]
    .mean()
)

summary_rows = []

for (release, feature_set), group in settings_mean.groupby(level=[0, 1]):
    default = per_split[(per_split.release == release) & (per_split.feature_set == feature_set)]

    for column, label in [("top2_accuracy", "Top-2 accuracy"), ("top3_accuracy", "Top-3 accuracy")]:
        values = group[column]
        summary_rows.append(
            dict(
                release=release,
                feature_set=feature_set,
                metric=label,
                lowest_setting_mean=values.min(),
                highest_setting_mean=values.max(),
                range_across_settings=values.max() - values.min(),
                default_forest_mean=default[column].mean(),
                split_to_split_sd=default[column].std(ddof=1),
            )
        )

sensitivity_summary = pd.DataFrame(summary_rows)

print("SPREAD ACROSS THE 12 SETTINGS COMPARED WITH THE SPLIT-TO-SPLIT SPREAD")
print("-" * 110)
print(sensitivity_summary.round(4).to_string(index=False))

SPREAD ACROSS THE 12 SETTINGS COMPARED WITH THE SPLIT-TO-SPLIT SPREAD
--------------------------------------------------------------------------------------------------------------
release feature_set         metric  lowest_setting_mean  highest_setting_mean  range_across_settings  default_forest_mean  split_to_split_sd
regions        full Top-2 accuracy               0.7216                0.7484                 0.0268               0.7107             0.0383
regions        full Top-3 accuracy               0.8415                0.8596                 0.0181               0.8415             0.0395
regions       no_ai Top-2 accuracy               0.7301                0.7648                 0.0347               0.7321             0.0452
regions       no_ai Top-3 accuracy               0.8493                0.8677                 0.0184               0.8658             0.0327
regions pure_record Top-2 accuracy               0.6733                0.7197                 0.0464              

## Results And Interpretation

All checks at the start passed: the six saved Top-1 models use exactly the predictors and settings defined in this notebook, and the seed-42 Top-1 results of all six cells are reproduced (differences at most 5.6e-17). The 50 splits have test partitions of 52 to 83 rows (mean 66.9). All values below are descriptive and carry no significance statement, because the splits overlap and the spread therefore does not represent the sampling variability of a new set of customers.

### How much the results vary between splits

Mean and standard deviation over the 50 splits (default forest, with Top-2 and Top-3 from the same forest):

| Cell | Top-1 accuracy | Macro F1 | Top-2 | Top-3 |
|---|---|---|---|---|
| source, full | 0.459 (0.047) | 0.351 (0.054) | 0.715 (0.050) | 0.847 (0.036) |
| regions, full | 0.460 (0.053) | 0.355 (0.067) | 0.711 (0.038) | 0.842 (0.040) |
| source, no AI | 0.475 (0.044) | 0.378 (0.058) | 0.730 (0.049) | 0.851 (0.040) |
| regions, no AI | 0.485 (0.046) | 0.390 (0.061) | 0.732 (0.045) | 0.866 (0.033) |
| source, pure record | 0.404 (0.046) | 0.325 (0.052) | 0.677 (0.046) | 0.818 (0.037) |
| regions, pure record | 0.404 (0.046) | 0.325 (0.052) | 0.677 (0.046) | 0.818 (0.037) |
| class-prior baseline | 0.198 (0.034) | 0.049 (0.007) | 0.395 (0.044) | 0.571 (0.049) |

- Every metric of every cell varies by about 4 to 5 percentage points (one standard deviation) from split to split. The 2.5th to 97.5th percentile of Top-1 accuracy of a single cell spans about 18 points (for example 0.370 to 0.556 for source, full).
- All six cells are far above the class-prior baseline (Top-1 about 0.40 to 0.48 against 0.20, and Top-3 about 0.82 to 0.87 against 0.57).
- The seed-42 results of the modelling notebooks are ordinary members of these distributions. Their Top-1 accuracy lies between 0.2 and 1.5 standard deviations above the 50-split mean, and the highest one, source with no AI (0.5424), is 1.5 standard deviations above its mean of 0.475, which explains why it looked like the best cell in the single split.

### Source release against anonymised release

Anonymised minus source, paired within each split (mean difference, with the 2.5th to 97.5th percentile of the 50 differences in brackets):

| Feature set | Top-1 accuracy | Macro F1 | Top-2 (final configuration) | Top-3 (final configuration) |
|---|---|---|---|---|
| full | +0.001 (-0.056 to +0.054) | +0.004 (-0.054 to +0.079) | +0.003 (-0.056 to +0.040) | -0.001 (-0.051 to +0.048) |
| no AI | +0.009 (-0.036 to +0.055) | +0.012 (-0.054 to +0.109) | +0.001 (-0.054 to +0.060) | -0.004 (-0.058 to +0.045) |
| pure record | 0.000 | 0.000 | 0.000 | 0.000 |

- For the pure-record features, whose predictors are identical in the two releases, the difference is exactly zero in all 50 splits for every metric. This confirms that the partitions, folds, seeds and code are the same for both releases.
- For the full and no-AI feature sets, which contain the eight generalised predictors, the mean differences are between -0.5 and +1.4 points and their sign varies from split to split (for Top-1 accuracy the difference is negative in 30 to 40 percent of the splits, zero in 12 to 16 percent and positive in 48 to 54 percent). Every percentile range contains zero, and the ranges (about 9 to 16 points wide) are of the size of the split-to-split variation of a single cell.
- The default forest and the final-configuration ranking forest (the setting of the source cell, used for both releases) lead to the same picture. Differences in Top-2 and Top-3 between releases in the single-split table of the modelling notebooks (for example -8.5 points for Top-2 with the full features) combine split variation with the different ranking-forest settings that the two searches selected. With identical settings, a Top-2 gap of -8.5 points is lower than the 2.5th percentile (-5.6 points) of the paired differences, so it is not a typical value.
- In this data the anonymised release gives results that are within the spread of ordinary split-to-split variation of the source release. This describes what was observed and does not show that anonymisation has no effect, particularly because the table is small (431 states).

### With and without the AI features

Full feature set minus no-AI feature set, paired within each split (with AI minus without AI):

| Release | Top-1 accuracy | Macro F1 | Top-2 | Top-3 |
|---|---|---|---|---|
| source | -0.016 (-0.079 to +0.047) | -0.027 (-0.161 to +0.039) | -0.015 (-0.099 to +0.059) | -0.005 (-0.057 to +0.053) |
| regions | -0.025 (-0.116 to +0.035) | -0.035 (-0.160 to +0.046) | -0.021 (-0.085 to +0.045) | -0.024 (-0.071 to +0.031) |

- On average the models scored 0.5 to 3.5 points lower with the AI features than without them, in both releases and for all five metrics. The difference is negative in 50 to 76 percent of the splits (Macro F1: 76 percent for source and 64 percent for regions) and positive in 12 to 36 percent, and every percentile range contains zero.
- The AI features therefore showed no advantage in these data. The result describes the fitted models and is not interpreted as a causal effect of the features, and part of the AI-derived information is time-stamped after the prediction time (see the modelling notebooks), which these comparisons do not address.

### Documented-record features only

Pure record minus full feature set, paired within each split:

| Release | Top-1 accuracy | Macro F1 | Top-2 | Top-3 |
|---|---|---|---|---|
| source | -0.055 (-0.177 to +0.048) | -0.025 (-0.149 to +0.075) | -0.038 (-0.123 to +0.027) | -0.029 (-0.089 to +0.053) |
| regions | -0.056 (-0.169 to +0.039) | -0.030 (-0.148 to +0.063) | -0.034 (-0.107 to +0.029) | -0.024 (-0.104 to +0.056) |

- The 12 documented-record predictors give on average 2.4 to 5.6 points lower scores than all 55 predictors, and the difference is negative in 58 to 80 percent of the splits. The loss is about one standard deviation of the split-to-split variation and is smaller when the selected ranking forest is used for Top-2 and Top-3 (1.2 to 2.6 points).
- With only these 12 predictors the models are still far above the class-prior baseline, so most of the usable signal is in the documented category counts and values.

### Dependence on the ranking-forest setting

The ranking forest was refitted with the 12 combinations of the values found in the six saved ranking forests (`min_samples_split` 2, 5 or 15, `min_samples_leaf` 1 or 6, and `max_depth` 7 or none). For each cell, the mean Top-2 differs by 2.7 to 4.6 points and the mean Top-3 by 1.2 to 4.0 points between the lowest and the highest of the 12 settings, which is about half to one standard deviation of the split-to-split variation. The largest ranges occur for the pure-record features (4.6 points for Top-2, 4.0 for Top-3). In three of the six cells the default forest gives a Top-2 about 1 point below the lowest of the 12 settings, and the final configuration gives 1.9 to 3.5 points higher Top-2 than the default forest and a Top-3 between 0.7 points lower and 2.2 points higher, but these settings were selected by searches on the seed-42 split, so such values can be somewhat optimistic. The comparisons between releases and between feature sets above give the same sign and a similar size with the default forest and with the final configuration.

### Limitations

- The table has 431 prediction states and 198 customers. The 50 splits are drawn from the same states, so the spread understates the uncertainty about new customers.
- The settings of the saved ranking forests were selected on the seed-42 split.
- The comparison measures predictive utility only and does not measure the privacy of the anonymised release.